# Bonus 06 — CrewAI role-based teams

CrewAI gives names to a team-shaped design: **agents** own roles and tools, **tasks** own deliverables, a **process** controls execution order, and a **crew** runs the whole unit.

You will build a release-readiness crew that retrieves one trusted change record, hands explicit evidence to a reviewer, returns typed outputs, rejects policy-breaking recommendations, and exposes tool calls, handoffs, token use, and cost.

> The lesson is not that every workflow needs multiple agents. The lesson is how to make role-based collaboration explicit — and how to recognize when ordinary code or a deterministic Flow is the better design.

## 1. Learn — a crew is an orchestration object

```mermaid
flowchart LR
    I["Input: change_id"] --> P["Sequential process"]
    P --> T1["Evidence task"]
    T1 --> A1["Evidence analyst agent"]
    A1 --> X["Trusted lookup tool"]
    X --> O1["Typed evidence output"]
    O1 -->|"explicit context"| T2["Review task"]
    T2 --> A2["Risk reviewer agent"]
    A2 --> G["Deterministic guardrail"]
    G --> O2["Typed release decision"]
```

The framework coordinates the arrows. Your application still decides which tools exist, which agent receives them, which task output becomes context, what policy is deterministic, and whether any later side effect is allowed.

### Four objects, four responsibilities

| Object | Owns | Does **not** prove |
|---|---|---|
| `Agent` | role, goal, backstory, model, available tools, iteration limits | identity, trust, or permission |
| `Task` | description, expected output, assigned agent, context, task-level tools, guardrails | that the answer is correct |
| `Process` | sequencing or manager-led coordination | durable state or safe side effects |
| `Crew` | the assembled agents and tasks, execution, aggregate usage | production authorization |

`role`, `goal`, and `backstory` shape a prompt. They are useful specialization cues, not security controls. Real authority belongs in software: tool exposure, credentials, schemas, policy checks, approval gates, and audit logs.

### Choose autonomy deliberately

| Need | Better starting point | Why |
|---|---|---|
| A few exact steps and deterministic rules | ordinary Python | least machinery; easiest to test |
| Open-ended work split across genuine specialties | CrewAI Crew | roles, tasks, context, delegation patterns |
| Auditable branching, state, retries, or external actions | CrewAI Flow | explicit event-driven control |
| Deterministic workflow with one exploratory pocket | Flow containing a Crew | control outside, autonomy where it earns its cost |

This lab uses a Crew because the handoff between two distinct responsibilities is the subject. A real release **approval** should remain a deterministic application or Flow decision.

### Use the lab's isolated environment

CrewAI 1.15.17 requires Python below 3.14 and resolves a large dependency graph, including versions of OpenAI and MCP that differ from the core course. From this directory run:

```bash
uv sync --locked
```

Then choose `bonus/06_crewai_role_based_teams/.venv/bin/python` as the VS Code notebook kernel. The lab still reads the repository-root `.env`; do not copy secrets here.

## 2. Do — build a visible two-role crew

In [ ]:
import json
import os
from importlib.metadata import distributions, version
from pathlib import Path
from typing import Literal

from crewai import Agent, Crew, LLM, Process, Task, TaskOutput
from crewai.tools import tool
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel

env_path = find_dotenv(usecwd=True)
assert env_path, "Repository-root .env not found"
load_dotenv(env_path)
ROOT = Path(env_path).parent
MODEL_DEFAULT = os.environ["MODEL_DEFAULT"]
PRICE_INPUT = float(os.environ["PRICE_INPUT_PER_MILLION"])
PRICE_CACHED_INPUT = float(os.environ["PRICE_CACHED_INPUT_PER_MILLION"])
PRICE_OUTPUT = float(os.environ["PRICE_OUTPUT_PER_MILLION"])
os.environ.setdefault("CREWAI_TRACING_ENABLED", "false")

print("CrewAI:", version("crewai"))
print("OpenAI SDK:", version("openai"))
print("Installed distributions in this environment:", len(list(distributions())))
print("Model from .env:", MODEL_DEFAULT)

### Start with trusted data and deterministic policy

The record is application data. It is not pasted into an agent's backstory. A tool exposes one record only when the model requests an exact ID. The application audit records the arguments and outcome.

In [ ]:
CHANGE_RECORDS = {
    "CHG-104": {
        "change_id": "CHG-104",
        "service": "orders-etl",
        "owner": "data-platform",
        "summary": "Add customer_tier and backfill 1,200,000 rows",
        "risk_signals": ["schema change", "backfill exceeds 100,000 rows"],
        "rollback": "restore view v41 and stop the backfill",
        "tests": ["unit passed", "staging passed"],
    },
    "CHG-205": {
        "change_id": "CHG-205",
        "service": "customer-export",
        "owner": "analytics-engineering",
        "summary": "Move the daily export schedule from 03:00 to 03:15 UTC",
        "risk_signals": ["schedule-only change"],
        "rollback": "restore the 03:00 UTC schedule",
        "tests": ["schedule validation passed"],
    },
}

tool_audit = []

@tool("lookup_change_record")
def lookup_change_record(change_id: str) -> str:
    """Return the trusted release record for one exact change ID."""
    record = CHANGE_RECORDS.get(change_id)
    tool_audit.append({
        "tool": "lookup_change_record",
        "arguments": {"change_id": change_id},
        "found": record is not None,
    })
    return json.dumps(record if record else {"error": "change not found"})

lookup_change_record.args_schema.model_json_schema()

### Type both handoff boundaries

Pydantic validates the shape of each task output. It does not prove that a risk level or owner is correct, so a later guardrail compares the decision with trusted application data.

In [ ]:
class ChangeEvidence(BaseModel):
    change_id: str
    service: str
    owner: str
    summary: str
    risk_signals: list[str]
    rollback_ready: bool


class ReleaseDecision(BaseModel):
    change_id: str
    risk_level: Literal["low", "medium", "high"]
    recommendation: Literal["proceed", "human_approval", "stop"]
    owner: str
    human_review_required: bool
    evidence: list[str]

### Configure one provider explicitly

CrewAI supports many providers, but this course keeps one provider and one environment-controlled model pin. For this model family, use `max_completion_tokens`, omit `temperature`, and pin `reasoning_effort="none"`.

In [ ]:
course_llm = LLM(
    model=f"openai/{MODEL_DEFAULT}",
    reasoning_effort="none",
    max_completion_tokens=700,
)

### Agents own responsibilities and capabilities

Only the evidence analyst receives the lookup tool. Delegation is disabled because the task graph already names the owners. Low iteration limits contain cost and looping.

In [ ]:
evidence_analyst = Agent(
    role="Change evidence analyst",
    goal="Retrieve one exact trusted change record and summarize only grounded release evidence.",
    backstory="You inspect data-platform changes. You never invent missing facts.",
    llm=course_llm,
    tools=[lookup_change_record],
    allow_delegation=False,
    max_iter=3,
    cache=False,
    verbose=False,
)

risk_reviewer = Agent(
    role="Release risk reviewer",
    goal="Apply release policy to supplied evidence and return a contained recommendation.",
    backstory="You provide analysis. You do not have authority to approve or execute a release.",
    llm=course_llm,
    allow_delegation=False,
    max_iter=2,
    verbose=False,
)

### Tasks own deliverables, dependencies, and validation

`context=[evidence_task]` is the visible handoff. The reviewer does not query the source again. A deterministic guardrail then checks the typed recommendation against trusted policy: schema changes or backfills above 100,000 rows require a person.

A failed guardrail sends feedback to the reviewer and allows one repair. It does not grant approval.

In [ ]:
evidence_task = Task(
    description=(
        "For change {change_id}, call lookup_change_record exactly once. "
        "Return its service, owner, summary, every risk signal, and whether a rollback plan exists. "
        "Use no outside facts."
    ),
    expected_output="A structured evidence record grounded only in the tool result.",
    agent=evidence_analyst,
    output_pydantic=ChangeEvidence,
)

guardrail_audit = []

def enforce_release_policy(result: TaskOutput):
    decision = result.pydantic or ReleaseDecision.model_validate_json(result.raw)
    record = CHANGE_RECORDS.get(decision.change_id)
    errors = []
    if record is None:
        errors.append("unknown change_id")
    else:
        requires_human = any(
            signal in {"schema change", "backfill exceeds 100,000 rows"}
            for signal in record["risk_signals"]
        )
        if decision.owner != record["owner"]:
            errors.append(f"owner must be {record['owner']}")
        if requires_human and not decision.human_review_required:
            errors.append("trusted policy requires human review")
        if requires_human and decision.recommendation == "proceed":
            errors.append("recommendation cannot be proceed without human approval")
    guardrail_audit.append({
        "change_id": decision.change_id,
        "accepted": not errors,
        "errors": errors,
    })
    return (not errors, result.raw if not errors else "; ".join(errors))

review_task = Task(
    description=(
        "Review the supplied evidence for {change_id}. Policy: schema changes or backfills above "
        "100,000 rows require human approval. Missing evidence means stop. Copy the trusted owner, "
        "and put the exact risk signals used in evidence."
    ),
    expected_output="A structured release decision with a grounded owner and evidence.",
    agent=risk_reviewer,
    context=[evidence_task],
    output_pydantic=ReleaseDecision,
    guardrail=enforce_release_policy,
    guardrail_max_retries=1,
)

### Prove the policy boundary before trusting a live run

A fabricated, well-typed but unsafe recommendation should fail. This direct call tests the deterministic predicate. During crew execution, CrewAI uses the same predicate and sends its feedback to the reviewer for at most one repair.

In [ ]:
unsafe_decision = ReleaseDecision(
    change_id="CHG-104",
    risk_level="high",
    recommendation="proceed",
    owner="data-platform",
    human_review_required=False,
    evidence=["schema change", "backfill exceeds 100,000 rows"],
)
unsafe_output = TaskOutput(
    description="Guardrail probe",
    expected_output="A contained decision",
    raw=unsafe_decision.model_dump_json(),
    pydantic=unsafe_decision,
    agent=risk_reviewer.role,
)
probe_ok, probe_feedback = enforce_release_policy(unsafe_output)
print("accepted:", probe_ok)
print("feedback:", probe_feedback)
assert probe_ok is False
assert "human review" in probe_feedback
guardrail_audit.clear()

### Inspect the wiring before execution

The assigned agent is not inferred at runtime. The context edge and tool boundary are inspectable application configuration.

In [ ]:
wiring = [
    {
        "task": "evidence",
        "agent": evidence_task.agent.role,
        "tools": [tool.name for tool in evidence_task.agent.tools],
        "context_tasks": len(evidence_task.context) if isinstance(evidence_task.context, list) else 0,
    },
    {
        "task": "review",
        "agent": review_task.agent.role,
        "tools": [tool.name for tool in review_task.agent.tools],
        "context_tasks": len(review_task.context) if isinstance(review_task.context, list) else 0,
    },
]
wiring

### Assemble a sequential crew and run it

Sequential means the tasks run in list order. A hierarchical process would add a manager model that plans and delegates; that is extra autonomy and extra model cost, not a free upgrade.

Jupyter already runs an event loop, so CrewAI 1.15.17 requires the asynchronous entry point here: `await crew.kickoff_async(...)`. This is the same async boundary introduced before frameworks in the core course.

In [ ]:
tool_audit.clear()
guardrail_audit.clear()

release_crew = Crew(
    agents=[evidence_analyst, risk_reviewer],
    tasks=[evidence_task, review_task],
    process=Process.sequential,
    memory=False,
    cache=False,
    tracing=False,
    verbose=False,
)

crew_result = await release_crew.kickoff_async(inputs={"change_id": "CHG-104"})
decision = crew_result.pydantic
decision

## 3. Observe — inspect the handoff, policy boundary, and cost

A fluent final answer is the least interesting artifact. Inspect what moved between roles and what the application checked.

In [ ]:
evidence_output, review_output = crew_result.tasks_output
print("HANDOFF TYPE:", type(evidence_output.pydantic).__name__)
print(evidence_output.pydantic.model_dump_json(indent=2))
print("\nFINAL TYPE:", type(review_output.pydantic).__name__)
print(review_output.pydantic.model_dump_json(indent=2))

### The tool and guardrail are separate application boundaries

The model chose the tool argument. Python executed the lookup. The reviewer proposed a decision. Python checked owner and human-review policy.

In [ ]:
print("TOOL AUDIT")
print(json.dumps(tool_audit, indent=2))
print("\nGUARDRAIL AUDIT")
print(json.dumps(guardrail_audit, indent=2))

assert tool_audit == [{
    "tool": "lookup_change_record",
    "arguments": {"change_id": "CHG-104"},
    "found": True,
}]
assert guardrail_audit[-1]["accepted"] is True
assert decision.owner == "data-platform"
assert decision.human_review_required is True
assert decision.recommendation != "proceed"

### Aggregate usage and estimate cost

CrewAI aggregates usage across tasks. One task is not necessarily one model request: a tool loop and conversion to a Pydantic output can each require additional requests. Inspect `successful_requests`; never estimate cost from the task count.

The class price estimate below remains the billing reference because it comes from the course `.env`, not a framework catalog. Cached prompt tokens are a subset of prompt tokens, not an extra charge.

In [ ]:
usage = release_crew.usage_metrics.model_dump()
prompt_tokens = usage.get("prompt_tokens", 0)
cached_prompt_tokens = usage.get("cached_prompt_tokens", 0)
completion_tokens = usage.get("completion_tokens", 0)
uncached_prompt_tokens = max(prompt_tokens - cached_prompt_tokens, 0)
estimated_cost = (
    uncached_prompt_tokens * PRICE_INPUT
    + cached_prompt_tokens * PRICE_CACHED_INPUT
    + completion_tokens * PRICE_OUTPUT
) / 1_000_000

print(json.dumps(usage, indent=2, default=str))
print(f"Estimated course-list cost: ${estimated_cost:.6f}")
print(f"Tool calls observed by the application: {len(tool_audit)}")

### What CrewAI bought — and what it did not

It bought explicit role/task objects, a readable context edge, process choices, typed task outputs, guardrail repair, callbacks and aggregate usage. It did **not** make roles trustworthy, make the handoff factual, authorize a release, persist resumable business state, or make several model requests cheaper than one.

Before adding an agent, ask: does this role have a distinct objective, capability, context, or evaluation criterion? If the only difference is a job title, keep one agent or use ordinary code.

## 4. Challenge — run a low-risk change through the same crew

Use `CHG-205`. Do not create new agents or change the trusted records. Reuse the crew with a new kickoff input, then verify that:

- the evidence analyst calls the lookup exactly once with `CHG-205`;
- the typed decision copies owner `analytics-engineering`;
- a schedule-only change can proceed without human review;
- the task handoff remains typed and the guardrail accepts it.

This is deliberately a reuse challenge. Creating a third 'schedule specialist' would add a title and a model call without adding a distinct capability.

In [ ]:
tool_audit.clear()
guardrail_audit.clear()

# TODO: await release_crew.kickoff_async with CHG-205 and keep the typed decision.
challenge_result = None
challenge_decision = None

In [ ]:
assert challenge_result is not None, "Run the crew and keep the CrewOutput"
assert isinstance(challenge_result.tasks_output[0].pydantic, ChangeEvidence)
assert isinstance(challenge_decision, ReleaseDecision)
assert tool_audit == [{
    "tool": "lookup_change_record",
    "arguments": {"change_id": "CHG-205"},
    "found": True,
}]
assert challenge_decision.change_id == "CHG-205"
assert challenge_decision.owner == "analytics-engineering"
assert challenge_decision.risk_level == "low"
assert challenge_decision.recommendation == "proceed"
assert challenge_decision.human_review_required is False
assert guardrail_audit[-1]["accepted"] is True
print("Challenge passed:", challenge_decision.model_dump())

## Takeaway

A Crew is useful when multiple model-driven responsibilities genuinely differ. Keep the collaboration inspectable: scope tools, name task owners, pass context explicitly, type outputs, put policy in deterministic code, cap iterations, and inspect usage. For exact business control, put the Crew inside ordinary software or a Flow — never confuse a role prompt with authority.